# Linear SVM and garment error analysis

Reproduces the UJ474 Phase 1 SVM experiment, exports the two standardized records, and creates focused garment-error figures.

In [3]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from aml.data import get_train_val_test_splits
from aml.features import get_features
from aml.models.svm import tune_svm, evaluate_final_svm, make_result_record, save_result
from aml.error_analysis import (plot_confusion_matrix, plot_garment_confusion_submatrix, extract_misclassifications, plot_misclassification_gallery)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'reports' / 'results'
FIGURES = ROOT / 'reports' / 'figures' / 'errors'
splits = get_train_val_test_splits(data_dir=ROOT / 'data' / 'raw')
FIGURES.mkdir(parents=True, exist_ok=True)
splits['class_names']

ModuleNotFoundError: No module named 'aml'

In [ ]:
all_runs = {}
for representation in ('pixels', 'hog'):
    X_train = get_features(splits['X_train'], representation)
    X_val = get_features(splits['X_val'], representation)
    X_train_full = get_features(splits['X_train_full'], representation)
    X_test = get_features(splits['X_test'], representation)
    best_C, tuning = tune_svm(X_train, splits['y_train'], X_val, splits['y_val'])
    model, metrics, latency_ms, parameter_count = evaluate_final_svm(X_train_full, splits['y_train_full'], X_test, splits['y_test'], best_C)
    record = make_result_record(representation, best_C, tuning, metrics, latency_ms, parameter_count, len(X_train), len(X_val), len(X_test))
    save_result(record, RESULTS)
    all_runs[representation] = {'model': model, 'metrics': metrics, 'tuning': tuning, 'record': record, 'predictions': model.predict(X_test)}
    display(pd.DataFrame(tuning))
    print(json.dumps(record, indent=2))

In [6]:
best_representation = max(all_runs, key=lambda name: all_runs[name]['record']['validation_macro_f1'])
run = all_runs[best_representation]
cm = run['metrics']['confusion_matrix']
plot_confusion_matrix(cm, splits['class_names'], f'Linear SVM ({best_representation}) — normalized confusion matrix', FIGURES / f'svm_{best_representation}_confusion_matrix.png')
plot_garment_confusion_submatrix(cm, splits['class_names'], save_path=FIGURES / f'svm_{best_representation}_upper_body_confusion.png')
plt.show()
print('Gallery representation:', best_representation)

NameError: name 'all_runs' is not defined

In [8]:
for true_class, predicted_class in [(6, 0), (2, 4)]:
    images, indices = extract_misclassifications(splits['y_test'], run['predictions'], splits['X_test'], true_class, predicted_class)
    plot_misclassification_gallery(images, indices, splits['class_names'][true_class], splits['class_names'][predicted_class], FIGURES / f'svm_{best_representation}_{true_class}_as_{predicted_class}_gallery.png')
    plt.show()

NameError: name 'extract_misclassifications' is not defined

## Interpretation

Read `reports/error_analysis.md` alongside the saved figures. The SVM's margin can only separate the information contained in pixels or fixed HOG bins; it cannot learn the missing spatial hierarchy needed for subtle collar, sleeve, and texture differences. A Phase 2 CNN should use the same test split and compare the Shirt row of its confusion matrix directly with this baseline.